In [1]:
import os

notebook_dir = os.path.dirname(os.path.abspath("day05_agent_v2.ipynb"))
root_dir = os.path.dirname(notebook_dir)
os.chdir(root_dir)

print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\Subramani Mokkala\Desktop\arc-agi3-research


In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import arc_agi

load_dotenv()

arc = arc_agi.Arcade(arc_api_key = os.getenv("ARC_API_KEY"))
envs = arc.get_environments()

print("Connected, environments loaded:", len(envs))

INFO:arc_agi.scorecard:Initialized ScorecardManager with idle_for=0:15:00 and max_open_for=3 days, 0:00:00
2026-06-16 17:00:51 | INFO | Successfully fetched 25 environment(s) from API
Connected, environments loaded: 25


In [4]:
env = arc.make(game_id = "ls20", save_recording = False)
obs = env.reset()

game = env._game
print(f"Player position: {game.gudziatsk.x}, {game.gudziatsk.y}")
print(f"Player shape: {game.fwckfzsyc}")
print(f"Player color: {game.hiaauhahz}")
print(f"Player rotation: {game.cklxociuu}")
print(f"Collectibles: {game.plrpelhym}")

2026-06-16 17:11:38 | INFO | Created new scorecard: c3952c84-0529-42dd-a2cd-58d9e717f29d
2026-06-16 17:11:40 | INFO | Successfully fetched metadata for game ls20
2026-06-16 17:11:40 | INFO | Successfully loaded game class Ls20 from environment_files\ls20\9607627b\ls20.py
Player position: 34, 45
Player shape: 5
Player color: 1
Player rotation: 3
Collectibles: [<arcengine.sprites.Sprite object at 0x000002E0A9791E90>]


In [5]:
for i, sprite in enumerate(game.plrpelhym):
    collectible_position = (sprite.x, sprite.y)
    required_shape = game.ldxlnycps[i]
    required_color = game.yjdexjsoa[i]
    required_rotation = game.ehwheiwsk[i]
    print(f"  collectible {i}: pos={collectible_position} requires shape={required_shape} color={required_color} rotation={required_rotation}")

  collectible 0: pos=(34, 10) requires shape=5 color=1 rotation=0


In [6]:
ACTION_UP = 1
ACTION_DOWN = 2
ACTION_LEFT = 3
ACTION_RIGHT = 4

In [7]:
def get_player_pos(game):
    return (game.gudziatsk.x, game.gudziatsk.y)

def get_player_state(game):
    return {
        "shape": game.fwckfzsyc,
        "color": game.hiaauhahz,
        "rotation": game.cklxociuu
    }

In [12]:
import random

from matplotlib.style import available

def choose_action(current, target, blocked):
    dx = target[0] - current[0]
    dy = target[1] - current[1]

    if abs(dx) > abs(dy):
        if dx < 0 and (ACTION_LEFT not in blocked):
            return ACTION_LEFT
        elif dx > 0 and (ACTION_RIGHT not in blocked):
            return ACTION_RIGHT
    else:
        if dy < 0 and (ACTION_UP not in blocked):
            return ACTION_UP
        elif dy > 0 and (ACTION_DOWN not in blocked):
            return ACTION_DOWN
    available = [a for a in [ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT] if a not in blocked]
    if not available:
        blocked.clear()
        available = [ACTION_UP, ACTION_DOWN, ACTION_LEFT, ACTION_RIGHT]
    return random.choice(available)


In [9]:
def is_stuck(pos_before, pos_after):
    return pos_before == pos_after

def get_level_plan(game):
    player_pos = get_player_pos(game)
    player_state = get_player_state(game)

    for i, sprite in enumerate(game.plrpelhym):
        collectible_pos = (sprite.x, sprite.y)
        required_shape = game.ldxlnycps[i]
        required_color = game.yjdexjsoa[i]
        required_rotation = game.ehwheiwsk[i]

        if (player_state["rotation"] != required_rotation):
            return (19, 30)
        if (player_state["color"] != required_color):
            return (49, 45)
        if (player_state["shape"] != required_shape):
            return None
        return collectible_pos

In [10]:
obs = env.reset()
step = 0
max_steps = 500
blocked = set()
prev_pos = None
last_action = None

print("starting agent")

while obs.state.value == "NOT_FINISHED" and step < max_steps:
    
    current = get_player_pos(env._game)
    target = get_level_plan(env._game)
    
    if target is None:
        break
    
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    
    action = choose_action(current, target, blocked)
    
    prev_pos = current
    last_action = action
    obs = env.step(action)
    step += 1
    
    if step % 20 == 0:
        print(f"step {step} | levels: {obs.levels_completed} | pos: {current}")

print(f"game ended after {step} steps")
print(f"final state: {obs.state.value}")
print(f"levels completed: {obs.levels_completed} / {obs.win_levels}")

starting agent
step 20 | levels: 0 | pos: (34, 30)
step 40 | levels: 0 | pos: (34, 40)
step 60 | levels: 0 | pos: (34, 30)
step 80 | levels: 0 | pos: (29, 25)
step 100 | levels: 1 | pos: (29, 35)
step 120 | levels: 1 | pos: (29, 35)
step 140 | levels: 1 | pos: (34, 35)
game ended after 149 steps
final state: GAME_OVER
levels completed: 1 / 7


In [13]:
obs = env.reset()

for i in range(105):
    current = get_player_pos(env._game)
    target = get_level_plan(env._game)
    if target is None:
        break
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    action = choose_action(current, target, blocked)
    prev_pos = current
    last_action = action
    obs = env.step(action)

print(f"levels completed: {obs.levels_completed}")
print(f"player pos: {get_player_pos(env._game)}")
print(f"player state: {get_player_state(env._game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

levels completed: 0
player pos: (29, 35)
player state: {'shape': 5, 'color': 1, 'rotation': 0}
collectible 0: pos=(14,40) requires shape=5 color=1 rotation=3


In [14]:
obs = env.reset()
prev_pos = None
last_action = None
blocked = set()

print(f"player pos: {get_player_pos(env._game)}")
print(f"player state: {get_player_state(env._game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

player pos: (29, 40)
player state: {'shape': 5, 'color': 1, 'rotation': 0}
collectible 0: pos=(14,40) requires shape=5 color=1 rotation=3


In [15]:
env = arc.make(game_id="ls20", save_recording=False)
obs = env.reset()
game = env._game
prev_pos = None
last_action = None
blocked = set()

print(f"player pos: {get_player_pos(game)}")
print(f"player state: {get_player_state(game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

2026-06-16 17:58:41 | INFO | Successfully fetched metadata for game ls20
player pos: (34, 45)
player state: {'shape': 5, 'color': 1, 'rotation': 3}
collectible 0: pos=(34,10) requires shape=5 color=1 rotation=0


In [18]:
env = arc.make(game_id="ls20", save_recording=False)
game = env._game
obs = env.reset()

step = 0
max_steps = 500
blocked = set()
prev_pos = None
last_action = None

print("starting agent")

while obs.levels_completed < 1 and obs.state.value == "NOT_FINISHED" and step < max_steps:
    current = get_player_pos(game)
    target = get_level_plan(game)
    if target is None:
        break
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    action = choose_action(current, target, blocked)
    prev_pos = current
    last_action = action
    obs = env.step(action)
    step += 1

print(f"Player pos: {get_player_pos(game)}")
print(f"Player state: {get_player_state(game)}")
for i, sprite in enumerate(game.plrpelhym):
    print(f"collectible {i}: pos=({sprite.x},{sprite.y}) requires shape={game.ldxlnycps[i]} color={game.yjdexjsoa[i]} rotation={game.ehwheiwsk[i]}")

2026-06-16 18:03:26 | INFO | Successfully fetched metadata for game ls20
starting agent
Player pos: (29, 40)
Player state: {'shape': 5, 'color': 1, 'rotation': 0}
collectible 0: pos=(14,40) requires shape=5 color=1 rotation=3


In [19]:
print(f"initial levels_completed: {obs.levels_completed}")
print(f"initial state: {obs.state.value}")

initial levels_completed: 1
initial state: NOT_FINISHED


In [20]:
for sprite in game.current_level._sprites:
    if sprite.tags:
        print(f"  pos=({sprite.x},{sprite.y}) tags={sprite.tags}")

  pos=(1,53) tags=['eqatonpohu']
  pos=(1,53) tags=['ghizzeqtoh']
  pos=(13,39) tags=['hoswmpiqkw']
  pos=(4,0) tags=['ihdgageizm']
  pos=(9,0) tags=['ihdgageizm']
  pos=(4,5) tags=['ihdgageizm']
  pos=(14,0) tags=['ihdgageizm']
  pos=(19,0) tags=['ihdgageizm']
  pos=(24,0) tags=['ihdgageizm']
  pos=(29,0) tags=['ihdgageizm']
  pos=(39,0) tags=['ihdgageizm']
  pos=(44,0) tags=['ihdgageizm']
  pos=(49,0) tags=['ihdgageizm']
  pos=(54,0) tags=['ihdgageizm']
  pos=(59,0) tags=['ihdgageizm']
  pos=(4,10) tags=['ihdgageizm']
  pos=(4,15) tags=['ihdgageizm']
  pos=(4,20) tags=['ihdgageizm']
  pos=(4,25) tags=['ihdgageizm']
  pos=(4,30) tags=['ihdgageizm']
  pos=(4,35) tags=['ihdgageizm']
  pos=(59,15) tags=['ihdgageizm']
  pos=(59,20) tags=['ihdgageizm']
  pos=(59,25) tags=['ihdgageizm']
  pos=(59,30) tags=['ihdgageizm']
  pos=(59,20) tags=['ihdgageizm']
  pos=(59,25) tags=['ihdgageizm']
  pos=(59,30) tags=['ihdgageizm']
  pos=(59,35) tags=['ihdgageizm']
  pos=(59,40) tags=['ihdgageizm']
  p

In [21]:
def find_sprite_by_tag(game, tag):
    for sprite in game.current_level._sprites:
        if tag in sprite.tags:
            return (sprite.x, sprite.y)
    return None

In [22]:
print(find_sprite_by_tag(game, "rhsxkxzdjz"))
print(find_sprite_by_tag(game, "rjlbuycveu"))
print(find_sprite_by_tag(game, "npxgalaybz"))

(49, 45)
(14, 40)
(15, 16)


In [23]:
def get_level_plan(game):
    player_pos = get_player_pos(game)
    player_state = get_player_state(game)

    for i, sprite in enumerate(game.plrpelhym):
        collectible_pos = (sprite.x, sprite.y)
        required_shape = game.ldxlnycps[i]
        required_color = game.yjdexjsoa[i]
        required_rotation = game.ehwheiwsk[i]

        if (player_state["rotation"] != required_rotation):
            return find_sprite_by_tag(game, "rhsxkxzdjz")
        if (player_state["color"] != required_color):
            return find_sprite_by_tag(game, "soyhouuebz")
        if (player_state["shape"] != required_shape):
            return find_sprite_by_tag(game, "ttfwljgohq")
        return collectible_pos

In [24]:
env = arc.make(game_id="ls20", save_recording=False)
game = env._game
obs = env.reset()
step = 0
max_steps = 500
blocked = set()
prev_pos = None
last_action = None

print("starting agent")

while obs.state.value == "NOT_FINISHED" and step < max_steps:
    current = get_player_pos(game)
    target = get_level_plan(game)
    if target is None:
        break
    if prev_pos is not None and is_stuck(prev_pos, current):
        blocked.add(last_action)
    else:
        blocked = set()
    action = choose_action(current, target, blocked)
    prev_pos = current
    last_action = action
    obs = env.step(action)
    step += 1

    if step % 20 == 0:
        print(f"step {step} | levels: {obs.levels_completed} | pos: {current} | target: {target}")

print(f"\ngame ended after {step} steps")
print(f"final state: {obs.state.value}")
print(f"levels completed: {obs.levels_completed} / {obs.win_levels}")

2026-06-16 18:09:15 | INFO | Successfully fetched metadata for game ls20
starting agent
step 20 | levels: 0 | pos: (34, 40) | target: (19, 30)
step 40 | levels: 0 | pos: (34, 40) | target: (19, 30)
step 60 | levels: 0 | pos: (34, 30) | target: (19, 30)
step 80 | levels: 0 | pos: (19, 25) | target: (34, 10)
step 100 | levels: 0 | pos: (34, 40) | target: (19, 30)
step 120 | levels: 0 | pos: (34, 35) | target: (19, 30)

game ended after 132 steps
final state: GAME_OVER
levels completed: 0 / 7
